# 👕👟👜 Fashion Item Classification using CNN (Fashion-MNIST)

**Project:** T-shirt, Shoes, Bag, Dress, etc. ki classification — Deep Learning CNN model k zariye.

**Dataset:** Fashion-MNIST (Keras built-in dataset)

**Level:** Intermediate — multiple Conv layers + Data Augmentation + Dropout + Batch Normalization

**Platform:** Google Colab

---

### 📋 Notebook ka structure (Steps):
1. Libraries import karna
2. GPU check karna (Colab settings)
3. Dataset load karna
4. Dataset explore karna (shapes, sample images)
5. Data preprocessing (normalize, reshape, one-hot encode)
6. Data Augmentation setup
7. CNN model banana
8. Model compile karna
9. Model train karna (with callbacks)
10. Training history visualize karna (accuracy/loss graphs)
11. Model evaluate karna (test data per)
12. Predictions check karna (sample images per)
13. Confusion Matrix aur Classification Report
14. Conclusion

---

> ⚠️ **Important Instruction (Colab Setup):**
> Yeh notebook run karne se pehle GPU enable karein:
> **Runtime → Change runtime type → Hardware accelerator → GPU (T4)** select karein, phir **Save** karein.
> Isse training fast hogi.


## Step 1: Required Libraries Import Karna

Yahan hum TensorFlow/Keras, NumPy, Matplotlib, aur Scikit-learn import kar rahe hain.
- **TensorFlow/Keras** → model banane aur train karne k liya
- **NumPy** → numerical operations k liya
- **Matplotlib/Seaborn** → graphs aur images dikhane k liya
- **Scikit-learn** → confusion matrix aur classification report k liya


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import confusion_matrix, classification_report

# Reproducibility k liya seed set karna (taake results consistent rahein)
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow Version:", tf.__version__)


## Step 2: GPU Check Karna

Yeh cell confirm karega ke Colab GPU use kar raha hai ya nahi. Agar "Num GPUs Available: 0" aaye, to upar wali instruction follow karke GPU enable karein.


In [ ]:
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))
print(tf.config.list_physical_devices('GPU'))


## Step 3: Fashion-MNIST Dataset Load Karna

Fashion-MNIST Keras k built-in datasets mein already maujood hai — internet se automatically download ho jayega, alag se kuch karne ki zaroorat nahi.

**Dataset Details:**
- 60,000 training images
- 10,000 testing images
- Har image **28x28 pixels**, grayscale
- 10 classes (neeche diye gaye hain)

**Classes:**
| Label | Class Name |
|-------|------------|
| 0 | T-shirt/top |
| 1 | Trouser |
| 2 | Pullover |
| 3 | Dress |
| 4 | Coat |
| 5 | Sandal |
| 6 | Shirt |
| 7 | Sneaker |
| 8 | Bag |
| 9 | Ankle boot |


In [ ]:
# Dataset load karna
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print("Training data shape:", x_train.shape)
print("Training labels shape:", y_train.shape)
print("Testing data shape:", x_test.shape)
print("Testing labels shape:", y_test.shape)


## Step 4: Dataset Explore Karna (Sample Images Dekhna)

Chaliye dataset ki kuch random images visualize karte hain taake pata chale data kaisa dikhta hai.


In [ ]:
plt.figure(figsize=(10, 10))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(x_train[i], cmap='gray')
    plt.xlabel(class_names[y_train[i]])
plt.suptitle("Fashion-MNIST Sample Images", fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
# Har class mein kitne images hain, yeh check karna (class balance dekhna)
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f"{class_names[u]:15s}: {c} images")


## Step 5: Data Preprocessing

Is step mein hum 3 kaam karenge:

1. **Normalization** — Pixel values (0-255) ko (0-1) range mein convert karna, taake model training fast aur stable ho.
2. **Reshaping** — CNN ko input 4D shape mein chahiye hoti hai: `(samples, height, width, channels)`. Hamara data grayscale hai, is liye channel = 1.
3. **One-Hot Encoding** — Labels (0-9) ko categorical format mein convert karna, kyunke hum `categorical_crossentropy` loss use karenge.


In [ ]:
# 1) Normalization: pixel values ko 0-1 range mein lana
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# 2) Reshape: CNN ko 4D input chahiye (samples, height, width, channels)
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

# 3) One-hot encoding: labels ko categorical banana
num_classes = 10
y_train_cat = keras.utils.to_categorical(y_train, num_classes)
y_test_cat = keras.utils.to_categorical(y_test, num_classes)

print("x_train shape after reshape:", x_train.shape)
print("y_train_cat shape after one-hot encoding:", y_train_cat.shape)


In [ ]:
# Training data ko train/validation mein split karna
from sklearn.model_selection import train_test_split

x_train, x_val, y_train_cat, y_val_cat = train_test_split(
    x_train, y_train_cat, test_size=0.1, random_state=42
)

print("Final Training samples:", x_train.shape[0])
print("Validation samples:", x_val.shape[0])
print("Test samples:", x_test.shape[0])


## Step 6: Data Augmentation Setup Karna

Data Augmentation se model ko thoda variation wala data milta hai (rotate, shift, zoom, etc.), jis se model **overfitting** kam karta hai aur **better generalize** karta hai unseen data par.

> Note: Hum sirf training data per augmentation apply karenge, validation/test data original rahega.


In [ ]:
datagen = ImageDataGenerator(
    rotation_range=10,        # images ko thori rotate karna (degrees)
    width_shift_range=0.1,    # horizontally shift karna
    height_shift_range=0.1,   # vertically shift karna
    zoom_range=0.1,           # zoom in/out
    shear_range=0.1,          # shear transformation
    horizontal_flip=False     # clothing items k liya flip avoid karna (kuch items asymmetric hote hain)
)

datagen.fit(x_train)
print("Data Augmentation generator ready hai.")


## Step 7: CNN Model Architecture Banana

Hamara CNN model in layers per mabni hoga:

- **Conv2D + BatchNormalization + Conv2D + MaxPooling + Dropout** (Block 1)
- **Conv2D + BatchNormalization + Conv2D + MaxPooling + Dropout** (Block 2)
- **Flatten**
- **Dense (fully connected) layers**
- **Output layer** (10 classes, softmax activation)

**Key concepts:**
- `Conv2D` → features (edges, patterns) detect karta hai
- `BatchNormalization` → training stable aur fast banata hai
- `MaxPooling2D` → image size reduce karta hai, important features rakhta hai
- `Dropout` → overfitting kam karta hai (random neurons off kar deta hai training k waqt)
- `Dense` → final classification decision leta hai


In [ ]:
model = models.Sequential([
    # ---- Block 1 ----
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(28, 28, 1)),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # ---- Block 2 ----
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # ---- Block 3 ----
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    # ---- Fully Connected Layers ----
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(num_classes, activation='softmax')
])

model.summary()


## Step 8: Model Compile Karna

- **Optimizer:** Adam (adaptive learning rate, fast convergence)
- **Loss Function:** Categorical Crossentropy (multi-class classification k liya)
- **Metrics:** Accuracy


In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model successfully compile ho gaya hai.")


## Step 9: Callbacks Define Karna

- **EarlyStopping** → agar validation loss improve nahi ho rahi to training rok dega (overfitting/time waste se bachne k liya)
- **ReduceLROnPlateau** → agar model plateau ho jaye to learning rate automatically kam kar dega


In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-5
)

callbacks_list = [early_stop, reduce_lr]


## Step 10: Model Train Karna

Hum `datagen.flow()` use karenge taake augmented data per training ho. Yeh step thora time lega (GPU per ~5-10 minutes, epochs k hisab se).

> Agar aap chahein to `epochs` ki value kam ya zyada kar sakte hain testing k liya.


In [ ]:
batch_size = 64
epochs = 25

history = model.fit(
    datagen.flow(x_train, y_train_cat, batch_size=batch_size),
    steps_per_epoch=len(x_train) // batch_size,
    epochs=epochs,
    validation_data=(x_val, y_val_cat),
    callbacks=callbacks_list,
    verbose=1
)


## Step 11: Training History Visualize Karna

Accuracy aur Loss ke graphs banayenge — training vs validation, taake overfitting/underfitting check ho sake.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Training Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss plot
axes[1].plot(history.history['loss'], label='Training Loss')
axes[1].plot(history.history['val_loss'], label='Validation Loss')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()


## Step 12: Model Evaluate Karna (Test Data Per)

Ab final test set per model ki performance check karte hain — yeh data model ne kabhi nahi dekha.


In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test_cat, verbose=0)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}  ({test_accuracy*100:.2f}%)")


## Step 13: Predictions Karna aur Sample Results Dekhna

Test set se kuch random images lekar dekhte hain ke model ne unhe sahi predict kiya ya nahi (green = correct, red = wrong).


In [ ]:
predictions = model.predict(x_test)
predicted_labels = np.argmax(predictions, axis=1)

plt.figure(figsize=(12, 12))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(x_test[i].reshape(28, 28), cmap='gray')

    true_label = y_test[i]
    pred_label = predicted_labels[i]
    color = 'green' if true_label == pred_label else 'red'

    plt.xlabel(f"P: {class_names[pred_label]}\nT: {class_names[true_label]}", color=color, fontsize=9)

plt.suptitle("Predictions (Green = Correct, Red = Wrong)", fontsize=16)
plt.tight_layout()
plt.show()


## Step 14: Confusion Matrix aur Classification Report

Confusion Matrix dikhata hai ke kis class ko model ne kis class k saath confuse kiya. Classification Report mein precision, recall, aur f1-score har class k liya milega.


In [ ]:
cm = confusion_matrix(y_test, predicted_labels)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
print("Classification Report:\n")
print(classification_report(y_test, predicted_labels, target_names=class_names))


## Step 15: Conclusion

✅ Humne Fashion-MNIST dataset per ek **Intermediate-level CNN model** train kiya jo T-shirt, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, aur Ankle Boot — total **10 classes** ko classify karta hai.

**Project mein kya use hua:**
- Multiple Conv2D blocks (32 → 64 → 128 filters)
- Batch Normalization (training stability)
- Dropout (overfitting control)
- Data Augmentation (better generalization)
- EarlyStopping & ReduceLROnPlateau callbacks
- Confusion Matrix & Classification Report (detailed evaluation)

**Aage improve karne k liya ideas:**
- Epochs aur batch size tune karna
- Aur Conv layers ya filters add karna
- Transfer Learning try karna (agar RGB images use karein)
- Hyperparameter tuning (Keras Tuner)

---
🎉 **Project complete!**
